In [ ]:
from pathlib import Path
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Direktori proyek
PROJECT_DIR = Path.cwd().parent

# Dataset Kalteng hasil filtering
KALTENG_DIR = (
    PROJECT_DIR
    / "datasets"
    / "processed"
    / "filtered"
    / "dataset-perkebunan-kalteng"
)

# Direktori train, validation, dan test
TRAIN_IMAGE_DIR = KALTENG_DIR / "train" / "images"
TRAIN_LABEL_DIR = KALTENG_DIR / "train" / "labels"

VAL_IMAGE_DIR = KALTENG_DIR / "valid" / "images"
VAL_LABEL_DIR = KALTENG_DIR / "valid" / "labels"

TEST_IMAGE_DIR = KALTENG_DIR / "test" / "images"
TEST_LABEL_DIR = KALTENG_DIR / "test" / "labels"

CLASS_NAMES = {
    0: "Unripe",
    1: "Ripe"
}

# Verifikasi direktori
for folder in [
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    VAL_IMAGE_DIR,
    VAL_LABEL_DIR,
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR
]:
    assert folder.exists(), f"Folder tidak ditemukan: {folder}"

print("Konfigurasi visual audit Kalteng berhasil.")
print("Train:", len(list(TRAIN_IMAGE_DIR.iterdir())))
print("Validation:", len(list(VAL_IMAGE_DIR.iterdir())))
print("Test:", len(list(TEST_IMAGE_DIR.iterdir())))
print("Class mapping:", CLASS_NAMES)

In [ ]:
def visualize_image(image_path, label_path, ax):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    ax.imshow(image)
    ax.set_title(image_path.name[:35], fontsize=9)
    ax.axis("off")

    if not label_path.exists():
        ax.set_title(f"{image_path.name} (LABEL MISSING)", color="red")
        return

    with open(label_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        x_center, y_center, box_width, box_height = map(float, parts[1:])

        # Konversi koordinat YOLO ke pixel
        x_center *= width
        y_center *= height
        box_width *= width
        box_height *= height

        x_min = x_center - box_width / 2
        y_min = y_center - box_height / 2

        rect = patches.Rectangle(
            (x_min, y_min),
            box_width,
            box_height,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        class_name = CLASS_NAMES.get(class_id, f"Unknown {class_id}")

        ax.text(
            x_min,
            max(0, y_min - 5),
            class_name,
            color="black",
            fontsize=9,
            bbox=dict(facecolor="lime", alpha=0.8)
        )


print("Fungsi visualisasi Kalteng berhasil dibuat.")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def show_random_samples(image_dir, label_dir, title, n=6, seed=42):
    image_paths = sorted([
        path for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ])

    random.seed(seed)
    selected = random.sample(image_paths, min(n, len(image_paths)))

    cols = 3
    rows = (len(selected) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = list(axes.flat) if hasattr(axes, "flat") else [axes]

    for ax, image_path in zip(axes, selected):
        label_path = label_dir / f"{image_path.stem}.txt"
        visualize_image(image_path, label_path, ax)

    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_random_samples(
    TRAIN_IMAGE_DIR,
    TRAIN_LABEL_DIR,
    "Kalteng - Random Training Samples",
    n=6
)

show_random_samples(
    TEST_IMAGE_DIR,
    TEST_LABEL_DIR,
    "Kalteng - Random Testing Samples",
    n=6,
    seed=100
)

In [ ]:
def calculate_bbox_area_ratio(image_dir, label_dir, split_name):
    results = []

    image_paths = [
        path for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    for image_path in image_paths:
        label_path = label_dir / f"{image_path.stem}.txt"

        with Image.open(image_path) as image:
            image_width, image_height = image.size

        with open(label_path, "r", encoding="utf-8") as file:
            lines = file.readlines()

        for box_index, line in enumerate(lines):
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            _, _, box_width, box_height = map(float, parts[1:])

            area_ratio = box_width * box_height

            results.append({
                "split": split_name,
                "image": image_path.name,
                "box_index": box_index,
                "class": CLASS_NAMES.get(class_id, "Unknown"),
                "width_ratio": box_width,
                "height_ratio": box_height,
                "area_ratio": area_ratio
            })

    return pd.DataFrame(results)


train_bbox_df = calculate_bbox_area_ratio(
    TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR, "train"
)

val_bbox_df = calculate_bbox_area_ratio(
    VAL_IMAGE_DIR, VAL_LABEL_DIR, "val"
)

test_bbox_df = calculate_bbox_area_ratio(
    TEST_IMAGE_DIR, TEST_LABEL_DIR, "test"
)

bbox_area_df = pd.concat(
    [train_bbox_df, val_bbox_df, test_bbox_df],
    ignore_index=True
)

print("Statistik rasio luas bounding box:")
display(
    bbox_area_df.groupby("split")["area_ratio"]
    .describe()
    .round(4)
)

In [ ]:
# Ambil 12 bounding box dengan luas terbesar
top_bbox = bbox_area_df.nlargest(12, "area_ratio")

print("12 bounding box dengan rasio luas terbesar:")
display(
    top_bbox[
        ["split", "image", "class", "width_ratio",
         "height_ratio", "area_ratio"]
    ].reset_index(drop=True).round(4)
)

# Direktori gambar berdasarkan split
SPLIT_IMAGE_DIRS = {
    "train": TRAIN_IMAGE_DIR,
    "val": VAL_IMAGE_DIR,
    "test": TEST_IMAGE_DIR
}

SPLIT_LABEL_DIRS = {
    "train": TRAIN_LABEL_DIR,
    "val": VAL_LABEL_DIR,
    "test": TEST_LABEL_DIR
}

fig, axes = plt.subplots(4, 3, figsize=(18, 22))
axes = axes.flatten()

for ax, (_, row) in zip(axes, top_bbox.iterrows()):
    image_path = SPLIT_IMAGE_DIRS[row["split"]] / row["image"]
    label_path = SPLIT_LABEL_DIRS[row["split"]] / f"{image_path.stem}.txt"

    visualize_image(image_path, label_path, ax)

    ax.set_title(
        f"{row['split'].upper()} | {image_path.name[:25]}\n"
        f"Area box: {row['area_ratio']:.1%}",
        fontsize=9
    )

plt.suptitle("Kalteng - 12 Largest Bounding Boxes", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
for _, row in top_bbox.iterrows():
    image_path = SPLIT_IMAGE_DIRS[row["split"]] / row["image"]
    label_path = SPLIT_LABEL_DIRS[row["split"]] / f"{image_path.stem}.txt"

    print("=" * 80)
    print("Split :", row["split"])
    print("Image :", image_path.name)
    print("Label :", label_path.name)

    with open(label_path, "r", encoding="utf-8") as file:
        for line in file:
            print("Raw annotation:", line.strip())

In [ ]:
def find_full_image_boxes(label_dir, split_name):
    results = []

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                class_id = int(parts[0])
                x, y, width, height = map(float, parts[1:])

                # Full-image box
                if (
                    abs(x - 0.5) < 1e-6
                    and abs(y - 0.5) < 1e-6
                    and abs(width - 1.0) < 1e-6
                    and abs(height - 1.0) < 1e-6
                ):
                    results.append({
                        "split": split_name,
                        "image": label_path.stem,
                        "class_id": class_id,
                        "line_number": line_number
                    })

    return results


full_image_results = []

for split, label_dir in [
    ("train", TRAIN_LABEL_DIR),
    ("val", VAL_LABEL_DIR),
    ("test", TEST_LABEL_DIR)
]:
    full_image_results.extend(
        find_full_image_boxes(label_dir, split)
    )

full_image_df = pd.DataFrame(full_image_results)

print("Total anotasi full-image:", len(full_image_df))

if not full_image_df.empty:
    print("\nDistribusi berdasarkan split:")
    display(full_image_df.groupby("split").size().to_frame("jumlah"))

    print("\nDistribusi berdasarkan kelas:")
    display(full_image_df.groupby(["split", "class_id"]).size().to_frame("jumlah"))

    print("\nDaftar anotasi:")
    display(full_image_df)
else:
    print("Tidak ditemukan anotasi full-image.")

In [ ]:
# Tampilkan seluruh gambar yang memiliki anotasi full-image
fig, axes = plt.subplots(5, 3, figsize=(18, 25))
axes = axes.flatten()

for ax, (_, row) in zip(axes, full_image_df.iterrows()):
    image_path = SPLIT_IMAGE_DIRS[row["split"]] / f"{row['image']}.jpg"
    label_path = SPLIT_LABEL_DIRS[row["split"]] / f"{row['image']}.txt"

    visualize_image(image_path, label_path, ax)

    ax.set_title(
        f"{row['split'].upper()} | {row['image'][:25]}\n"
        f"Class ID: {row['class_id']}",
        fontsize=9
    )

for ax in axes[len(full_image_df):]:
    ax.axis("off")

plt.suptitle("Kalteng - Audit 15 Full-Image Annotations", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
suspect_images = [
    ("train", "frame_6929_png.rf.0d507b7c00f67e76f6aa1c6f7875b3b6"),
    ("train", "frame_6968_png.rf.b22959b4a993ea1dbb6e3ff0c310cbb7"),
    ("test", "frame_001357_png_jpg.rf.58d7fd5422f24db965cf2b"),
]

for split, stem in suspect_images:
    image_dir = SPLIT_IMAGE_DIRS[split]
    label_dir = SPLIT_LABEL_DIRS[split]

    matches = [
        p for p in image_dir.glob("*")
        if p.is_file() and p.stem == stem
    ]

    if not matches:
        print(f"Gambar tidak ditemukan: {split}/{stem}")
        continue

    image_path = matches[0]
    label_path = label_dir / f"{stem}.txt"

    print(f"\n{split.upper()} | {image_path.name}")
    print("Resolusi:", Image.open(image_path).size)

    fig, ax = plt.subplots(figsize=(9, 7))
    visualize_image(image_path, label_path, ax)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cari gambar test berdasarkan sebagian nama file
matches = [
    p for p in SPLIT_IMAGE_DIRS["test"].glob("*")
    if "frame_001357" in p.name
]

print("Jumlah gambar ditemukan:", len(matches))

for image_path in matches:
    label_path = SPLIT_LABEL_DIRS["test"] / f"{image_path.stem}.txt"

    print("\nGambar:", image_path.name)
    print("Label:", label_path.name)

    image = Image.open(image_path).convert("RGB")
    print("Resolusi:", image.size)

    fig, ax = plt.subplots(figsize=(9, 7))
    visualize_image(image_path, label_path, ax)
    plt.tight_layout()
    plt.show()